In [ ]:
print("Hello world")

In [ ]:
!unzip -q "/content/asl_data.zip" -d "/content/asl_dataset"

In [ ]:
COUNT = 500
AUGMENTATIONS = 2
TRAIN = '/content/asl_dataset/dataset'

We want image embeddings for both EDA and classification, using ResNeXt and data augmentation for Random Forest training.

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
import numpy as np

DEVICE = torch.device('cpu')
OUTPUT_SIZE = 2048

model = models.resnext50_32x4d(weights=models.ResNeXt50_32X4D_Weights.IMAGENET1K_V2)
extraction_layer = model._modules.get('avgpool')
model.to(DEVICE)
model.eval()

scaler = transforms.Resize((224, 224))
normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
to_tensor = transforms.ToTensor()
augmenter = transforms.RandomAffine(degrees=12, translate=(0.08, 0.08), scale=(0.85, 1.15),
                                     interpolation=transforms.InterpolationMode.BILINEAR, fill=(255, 255, 255))

def get_vec(arg, model, extraction_layer, augmented=False):
    image = normalize(to_tensor(scaler(augmenter(arg) if augmented else arg))).unsqueeze(0).to(DEVICE)
    result = torch.zeros(1, OUTPUT_SIZE, 1, 1)
    def copy_data(m, i, o):
        result.copy_(o.data)
    hooked = extraction_layer.register_forward_hook(copy_data)
    with torch.no_grad():
        model(image)
    hooked.remove()
    return result

Let's load up (a sample of) our data, including image embeddings and thumbnails. This is going to take a while, because we have so many classes.

In [ ]:
import arrow
import base64
import pandas as pd
from glob import iglob
from io import BytesIO
from os.path import basename
from PIL import Image

THUMBNAIL_SIZE = (64, 64)

def embed(model, filename: str, augmented=False):
    with Image.open(fp=filename, mode='r') as image:
        return get_vec(arg=image.convert('RGB'), model=model, extraction_layer=extraction_layer,
                       augmented=augmented).numpy().reshape(OUTPUT_SIZE,)

def flatten(arg):
    return [x for xs in arg for x in xs]

def png(filename: str) -> str:
    with Image.open(fp=filename, mode='r') as image:
        buffer = BytesIO()
        image.resize(size=THUMBNAIL_SIZE).convert('RGB').save(buffer, format='png')
        return 'data:image/png;base64,' + base64.b64encode(buffer.getvalue()).decode()

def get_picture_from_glob(arg: str, tag: str,) -> list:
    time_get = arrow.now()
    result = [pd.Series(data=[tag, basename(input_file), input_file, embed(model=model, filename=input_file), png(filename=input_file)],
                        index=['tag', 'name', 'path', 'value', 'png'])
        for index, input_file in enumerate(list(iglob(pathname=arg))) if index < COUNT]
    print('encoded {} rows of {} in {}'.format(len(result), tag, arrow.now() - time_get))
    return result

time_start = arrow.now()
train_dict = {basename(folder): folder + '/*.*' for folder in iglob(TRAIN + '/*')}
train_df = pd.DataFrame(data=flatten(arg=[get_picture_from_glob(arg=value, tag=key) for key, value in train_dict.items()]))
print('done in {}'.format(arrow.now() - time_start))

Next let's use TSNE to get x/y coordinates from our image embeddings.

In [ ]:
from sklearn.manifold import TSNE

train_reducer = TSNE(random_state=2025, verbose=True, n_jobs=1, perplexity=20.0, init='pca')
train_df[['x', 'y']] = train_reducer.fit_transform(X=train_df['value'].apply(func=pd.Series))

Now we're ready to plot.

In [ ]:
from bokeh.models import ColumnDataSource
from bokeh.models import HoverTool

from bokeh.plotting import figure
from bokeh.plotting import output_notebook
from bokeh.plotting import show
from bokeh.palettes import Turbo256
from bokeh.transform import factor_cmap

output_notebook()

datasource = ColumnDataSource(train_df[['png', 'tag', 'x', 'y']].sample(n=2000, random_state=2025))
factor_count = max(train_df['tag'].nunique(), 3)
indices = np.linspace(0, len(Turbo256)-1, factor_count, dtype=int)
palette = [Turbo256[index] for index in indices]
mapper = factor_cmap(field_name = 'tag', palette=palette, factors=train_df['tag'].unique().tolist(), start=0, end=factor_count-1, )

plot_figure = figure(title='TSNE projection: skeletons', width=1000, height=800, tools=('pan, wheel_zoom, reset'))

plot_figure.add_tools(HoverTool(tooltips="""
<div>
    <div>
        <img src='@png' style='float: left; margin: 5px 5px 5px 5px'/>
    </div>
    <div>
        <span style='font-size: 18px'>@tag</span>
    </div>
</div>
"""))

plot_figure.scatter(x='x', y='y', source=datasource, line_alpha=0.6, fill_alpha=0.6, size=8, color=mapper)
show(plot_figure)

The embeddings are used to train a RandomForestClassifier. The train/test split is performed before augmentation.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

train_indices, test_indices = train_test_split(
    np.arange(len(train_df)), test_size=0.25, random_state=2025, stratify=train_df['tag'])
X_train = np.stack(train_df.iloc[train_indices]['value'].to_numpy())
y_train = train_df.iloc[train_indices]['tag'].tolist()
X_test = np.stack(train_df.iloc[test_indices]['value'].to_numpy())
y_test = train_df.iloc[test_indices]['tag'].tolist()

for index in range(AUGMENTATIONS):
    augmented_features = np.stack([
        embed(model=model, filename=filename, augmented=True)
        for filename in train_df.iloc[train_indices]['path']
    ])
    X_train = np.concatenate([X_train, augmented_features], axis=0)
    y_train += train_df.iloc[train_indices]['tag'].tolist()

randomforest = RandomForestClassifier(n_estimators=160, max_depth=24, min_samples_leaf=2,
                                      random_state=2025, n_jobs=-1).fit(X_train, y_train)
print('accuracy: {:5.4f}'.format(accuracy_score(y_true=y_test, y_pred=randomforest.predict(X=X_test))))
print('f1: {:5.4f}'.format(f1_score(average='weighted', y_true=y_test, y_pred=randomforest.predict(X=X_test))))
print(classification_report(y_true=y_test, y_pred=randomforest.predict(X=X_test)))

Let's inspect the probabilities returned by Random Forest on the untouched test subset.

In [ ]:
import matplotlib.pyplot as plt
import warnings
from seaborn import histplot
from seaborn import scatterplot

warnings.filterwarnings('ignore', category=FutureWarning, module='seaborn')

plot_df = train_df.iloc[test_indices][['x', 'y']].copy()
plot_df['probability'] = np.max(randomforest.predict_proba(X=X_test), axis=1)

fig, ax = plt.subplots(ncols=2, figsize=(12, 6))
scatterplot(ax=ax[0], data=plot_df, x='x', y='y', hue='probability', palette='viridis')
ax[0].set(xlabel=None)
ax[0].set(ylabel=None)
ax[0].set(xticklabels=[])
ax[0].set(yticklabels=[])
ax[0].tick_params(axis='both', which='both', length=0)
histplot(ax=ax[1], data=plot_df, x='probability', bins=20)

Random Forest averages the class probabilities of its decision trees. Evaluation uses only non-augmented test images.

In [ ]:
import joblib
joblib.dump(randomforest, 'randomforest.pkl')